# RD02 — Latent-regime tournament

Three-state Student-t HMM vs causal age-dependent semi-Markov overlay, same-origin observable tree and continuous return-skew controls. **Historical exploratory only**; no AM-T-HAR-Blend density claim.

In [ ]:
import os,sys,time,json,subprocess,tempfile,gc
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Run this notebook from repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import latent_regime_tournament as m
SYNTHETIC=os.getenv('RD02_SYNTHETIC','0')=='1'
if SYNTHETIC:
    temp=tempfile.TemporaryDirectory(prefix='rd02-')
    OUT=Path(temp.name)/'02'
    cfg=m.Spec(em_iterations=3,bootstrap_reps=25)
else:
    root=os.getenv('DERIBIT_DATA_ROOT')
    if not root:raise RuntimeError('DERIBIT_DATA_ROOT missing from .env or environment')
    STORE=Path(root)/'research'/'cs04a_feature_store'
    OUT=ROOT/'notebooks'/'Regime_Discovery'/'02'
    cfg=m.Spec()
OUT.mkdir(parents=True,exist_ok=True)
print(m.VERSION,'synthetic=',SYNTHETIC,'output=',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
check=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_latent_regime_tournament.py')],
                  cwd=str(ROOT),env=env,capture_output=True,text=True)
print(check.stdout[-1500:]);print(check.stderr[-500:]);assert check.returncode==0,'RD02 tests failed; stop'


In [ ]:
panels={};excluded={}
if SYNTHETIC:
    sys.path.insert(0,str(ROOT/'tests'))
    from test_latent_regime_tournament import synthetic_panel
    panels={'BTC':synthetic_panel(days=80,seed=cfg.seed)}
else:
    required=list(dict.fromkeys(['state_time','return_5m','feature_ready','log_rv24h',
                           *m.STATE_FEATURES,*m.RETURN_FEATURES,'ret_5m','ret_15m','ret_30m']))
    for asset in cfg.assets:
        path=STORE/f'{asset}_features_and_forward_magnitude.parquet'
        if not path.is_file():excluded[asset]=f'Missing: {path}';continue
        try:
            panels[asset]=pd.read_parquet(path,columns=required)
            print(asset,len(panels[asset]),'rows loaded')
        except Exception as exc:excluded[asset]=f'{type(exc).__name__}: {exc}'
    if not {'BTC','ETH'}.issubset(panels):
        raise RuntimeError(f'Primary assets missing: {excluded}')
assert panels,excluded


In [ ]:
results={};coverage=[]
for asset,panel in panels.items():
    start=time.perf_counter()
    result=m.scan_asset(panel,asset,cfg)
    results[asset]=result
    coverage.append({'asset':asset,'complete_days':result['complete_days'],
                     'asset_fold_horizon_scores':len(result['scores']),
                     'observable_tree_definitions':len(result['definitions']),
                     'elapsed_seconds':round(time.perf_counter()-start,1)})
    print(asset,'finished in',round(time.perf_counter()-start,1),'s')
    del panel;gc.collect()
coverage=pd.DataFrame(coverage)
display(coverage)


In [ ]:
out=m.consolidate(results,cfg,synthetic=SYNTHETIC)
out['manifest']['excluded']=excluded
print('Mean NLL improvement against fitted continuous-skew benchmark (negative better)')
display(out['comparison'].sort_values(['horizon_min','delta_nll_vs_continuous_skew']))
print('Primary 30m NLL, paired day-bootstrap (historical exploratory):')
display(out['bootstrap'].query('horizon_min==30').head(50))
print('State diagnostics')
display(out['states'].head(18))
print('30-minute validated return by causally filtered HMM state')
display(out['validation_states'].head(20))
print('Decision:',json.dumps(out['decision'],indent=2))


In [ ]:
mapping={
 '02_scores.csv':'scores','02_comparison_summary.csv':'comparison',
 '02_paired_day_bootstrap.csv':'bootstrap','02_state_diagnostics.csv':'states',
 '02_return_response.csv':'responses','02_transition_matrix.csv':'transitions',
 '02_duration_hazards.csv':'hazards','02_validation_state_behavior.csv':'validation_states',
 '02_observable_tree_definitions.csv':'definitions'
}
for name,key in mapping.items():out[key].to_csv(OUT/name,index=False)
# Aggregate durations only, explicitly distinguish right-censored episodes.
ep=out['episodes'].groupby(['asset','fold','state','censored_at_segment_end'],as_index=False).agg(
   episodes=('duration_blocks','size'),mean_duration_blocks=('duration_blocks','mean'),
   median_duration_blocks=('duration_blocks','median'),p90_duration_blocks=('duration_blocks',lambda x:x.quantile(.9)))
ep.to_csv(OUT/'02_filtered_episode_summary.csv',index=False)
coverage.to_csv(OUT/'02_coverage.csv',index=False)
(OUT/'02_manifest.json').write_text(json.dumps(out['manifest'],indent=2,default=str),encoding='utf-8')
(OUT/'02_decision.json').write_text(json.dumps(out['decision'],indent=2,default=str),encoding='utf-8')
assert all((OUT/name).is_file() for name in [*mapping,'02_filtered_episode_summary.csv',
                                       '02_coverage.csv','02_manifest.json','02_decision.json'])
print('Wrote',len(mapping)+4,'aggregates to',OUT,'synthetic=',SYNTHETIC)
print('Never commit synthetic outputs or origin-level sensitive data.')


## Interpretation limits

The semi-Markov extension is a **two-stage fitted hazard overlay**, not joint HSMM inference. Latent IDs are not stable labels between folds. All forecasts condition on current known features; Viterbi is used only on train for duration fit. Model uses trailing variance proxy, not AM-T-HAR-Blend. A negative 30m NLL delta on studied history is not independent evidence.